# Workspace
エージェント機能を持つチャットボットのサンプル。 mlflow と連携

In [ ]:
import mlflow
import zoneinfo

tz_info = zoneinfo.ZoneInfo("Asia/Tokyo")
mlflow.set_experiment("agent-rag")

# Playground

In [ ]:
import datetime
import importlib
from mlflow.types.responses import Message, ResponsesAgentRequest

import agent

importlib.reload(agent)

ymd = datetime.datetime.now(tz=tz_info).strftime("%Y%m%d_%H%M%S")
prompt = """
Databricks とその周辺の製品に関しての直近 (2025/02-) のニュースを振り返るため、まとめてください。
作業はタスクリストを作成し、上から順に step by step で処理していってください。

<instruction>
1. 対象となる期間を指定し検索。ニュースを取得
2. ニュースを製品毎に整理
3. 指定期間を通し特筆すべき動向を3つ以上抽出
4. ここまで調べ整理した内容を入力にレイアウトを整理した上で最終成果物を書出し
5. 最終成果物を読み、Output Requirement を満たすかを確認。満たせば処理完了。最終成果物を出力
</instruction>

<background>
現在日時: 2026/01/12

読み手は ITの専門的知識を持つ人間を想定。指定期間のIT業界の動向を振り返る必要が有る。
また、業界の動向のみならず同時期に発生したイベントやニュースも見る事で、広い視野で業界の動向を知る。
</background>

## Tool Guideline
- **search_news_history**: 月毎のニュースを取得するのに使用。対象月 (yyyy/MM) を指定し検索する

## Output Requirement
- 製品・作品単位でまとめてください
- 指定期間中の動向を複数の観点でまとめてください

例を以下に記載する。

```markdown
# 製品名A
- {製品Aのニュース1。100文字前後} ({yyyy/MM} ニュース日付を記載)
- {製品Aのニュース2。100文字前後} ({yyyy/MM} ニュース日付を記載)

# 製品名B
- {製品Bのニュース3。100文字前後} ({yyyy/MM} ニュース日付を記載)
- {製品Bのニュース4。100文字前後} ({yyyy/MM} ニュース日付を記載)

# IT業界外のニュース
- {他ニュース1。100文字前後} ({yyyy/MM} ニュース日付を記載)
- {他ニュース2。100文字前後} ({yyyy/MM} ニュース日付を記載)

# 期間中の動向 (対象製品周辺)
- 観点1
- 観点2

# 期間中の動向 (IT業界外)
- 観点1
- 観点2
```
"""

with mlflow.start_run(run_name=f"dev_{ymd}"):
    # 推論 (逐次)
    res = agent.agent_wrapped.predict(
        ResponsesAgentRequest(
            input=[
                Message(role="user", content=prompt),
            ]
        )
    )
    print(res.output)

In [ ]:
print(len(res.output))
print(res.output[-1].content[0]["text"])

# Test

In [ ]:
import datetime
import importlib
import agent
import evaluate

# テストデータを作成
eval_dataset = [
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "東京都の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "東京都の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "横浜市の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "横浜市の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "群馬県の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "群馬県の天気は豪雨です。"},
    },
]

ymd = datetime.datetime.now(tz=tz_info).strftime("%Y%m%d_%H%M%S")
with mlflow.start_run(run_name=f"dev_{ymd}"):
    # モデルを評価
    importlib.reload(agent)
    res = evaluate.eval_responses(agent.agent_wrapped, eval_dataset)
    print(res)